# Phase 2 — S3 premise permutations

Separate notebook for the next experiment. It uses four Lean-certified three-premise families and all six premise-order permutations. Run one cell at a time from top to bottom.


## 1. Recover `/content` safely


In [ ]:
import os, shutil, subprocess
os.chdir('/content')
repo = '/content/proof-path-invariance'
if os.path.exists(repo):
    shutil.rmtree(repo)
print('cwd:', os.getcwd())


## 2. Clone the latest repository


In [ ]:
subprocess.run([
    'git', 'clone',
    'https://github.com/Kairose-master/proof-path-invariance.git',
    '/content/proof-path-invariance'
], check=True, cwd='/content')
os.chdir('/content/proof-path-invariance')
print('cwd:', os.getcwd())


## 3. Install runner dependencies


In [ ]:
!python3 -m pip install -q -r requirements-runner.txt


## 4. Generate the frozen S3 benchmark


In [ ]:
!python3 scripts/generate_s3_benchmark.py --out /tmp/s3_v0.jsonl


## 5. Validate all 128 cases and six permutations per case


In [ ]:
!python3 scripts/validate_s3_benchmark.py /tmp/s3_v0.jsonl


## 6. Verify the frozen benchmark hash


In [ ]:
!python3 scripts/verify_s3_lock.py /tmp/s3_v0.jsonl


Continue only if the final line is `S3 benchmark lock verified`.


## 7. Run Pythia-70M on all 768 prompts


In [ ]:
!mkdir -p results
!python3 scripts/run_hf_s3_margin.py --prompts /tmp/s3_v0.jsonl --out results/pythia70m-step143000-s3-v0.jsonl --run-id pythia70m-step143000-s3-v0


The raw result file is exclusive-create. Do not rerun this cell against the same output path after collecting data.


## 8. Score permutation sensitivity


In [ ]:
!python3 scripts/score_s3.py results/pythia70m-step143000-s3-v0.jsonl


Send the scorer JSON back to ChatGPT before changing the benchmark or analysis. Save `results/pythia70m-step143000-s3-v0.jsonl` before the Colab runtime closes.


## 9. Download raw result and scorer output


In [ ]:
from google.colab import files
!python3 scripts/score_s3.py results/pythia70m-step143000-s3-v0.jsonl > /content/phase2_s3_score.json
files.download('/content/proof-path-invariance/results/pythia70m-step143000-s3-v0.jsonl')
files.download('/content/phase2_s3_score.json')


Do not close the runtime until both download prompts have appeared. The raw JSONL is the primary artifact needed for Phase 2.5.
